In [1]:
import requests
import pandas as pd
import io
import os

In [2]:
if os.path.exists('istat_raw.csv'):
    print("file già esistente, la chiamata API è stata skippata")
else:
    url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
    headers_csv = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

    r = requests.get(url, headers=headers_csv, timeout=120)
    print(r.status_code)

    if r.status_code == 200:
        with open('istat_raw.csv', 'w', encoding='utf-8') as f:
            f.write(r.text)
        print("Salvato su disco, non serve richiamare l'API di nuovo")

# l'API SDMX di ISTAT ha un limite di 5 chiamate al minuto: se lo superi,
# l'IP viene bloccato per 24-48 ore (un meccanismo un po' becero, a dirla
# come l'ha detto il prof)

# dato che i dati storici sugli incidenti non cambiano in tempo reale,
# non ha senso interrogare l'API ogni volta che eseguo il notebook.
# per questo aggiungo un controllo con if/else: se il file istat_raw.csv
# esiste già sul disco, salto del tutto la chiamata API; altrimenti,
# solo la prima volta, faccio la chiamata e appena arriva la risposta
# la salvo subito su disco. così, anche rieseguendo il notebook da capo
# più volte (es. con Restart & Run All), non rischio mai di superare il
# limite di chiamate al minuto, e mostro anche un po' di pietà per
# l'infrastruttura non esattamente performante di ISTAT

file già esistente, la chiamata API è stata skippata


In [3]:
df = pd.read_csv('istat_raw.csv')
print(df.info())
df

# visualizzo le informazioni del dataframe, ci sono 573.552 righe e 16 colonne ma 9 di queste
# sono completamente vuote (0 non-null):
# OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT,
# NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, UNIT_MULT
# quindi le rimuoverò in fase di pulizia

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573547,IT1:41_983(1.0),A,111107,ROADACC,9,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573548,IT1:41_983(1.0),A,111107,ROADACC,9,2021,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573549,IT1:41_983(1.0),A,111107,ROADACC,9,2022,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573550,IT1:41_983(1.0),A,111107,ROADACC,9,2023,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(df['REF_AREA'].astype(str).str.len().unique())
print(df['DATA_TYPE'].unique())
print(df['RESULT'].unique())
print(df['DATAFLOW'].unique())
print(df['FREQ'].unique())

# ho controllato quante cifre hanno i codici REF_AREA convertendoli in stringa
# e misurandone la lunghezza: sono uscite lunghezze diverse (4, 5, 6) invece
# di un valore uniforme. Questo succede perché la colonna è stata letta da
# pandas come numero (int64), e i numeri non mantengono gli zeri davanti,
# quindi un codice come 001001 viene letto e salvato come 1001 (4 cifre).
# i codici comune ISTAT dovrebbero invece essere sempre a 6 cifre, quindi
# l'ho corretto

# ho controllato anche i valori unici di DATA_TYPE, RESULT, DATAFLOW e FREQ per capire
# cosa sono e cosa contengono queste colonne prima di
# usarle nell'analisi:
# - DATA_TYPE ha 2 valori: KILLINJ, ROADACC;
# - RESULT ha 3 valori: F, M, 9;
# - DATAFLOW e FREQ hanno un solo valore, rispettivamente 'IT1:41_983(1.0)' e 'A'
# da questi nomi non è ancora chiaro il significato esatto di alcuni valori (es. cosa
# rappresenta il valore 9), dopo controllerò, ma posso dire già con certezza che nè DATAFLOW nè FREQ
# hanno dei valori sensati perchè entrambi ne hanno solo uno per tutte le righe, quindi
# non apportano alcuna informazione utile

[4 5 6]
<StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
<StringArray>
['F', 'M', '9']
Length: 3, dtype: str
<StringArray>
['IT1:41_983(1.0)']
Length: 1, dtype: str
<StringArray>
['A']
Length: 1, dtype: str


In [5]:
df['REF_AREA'] = df['REF_AREA'].astype(str).str.zfill(6)

print(df['REF_AREA'].head(10))
print(df['REF_AREA'].astype(str).str.len().unique())

# prima avevo trovato che REF_AREA aveva lunghezze diverse (4, 5, 6 cifre)
# perché pandas l'aveva letta come numero, perdendo gli zeri davanti ai
# codici comune. qui la converto in stringa con zfill(6), che aggiunge
# zeri a sinistra finché la stringa non arriva a 6 caratteri, così i
# codici tornano tutti uniformi e nel formato giusto e uguale per tutti i record

# controllo che il fix abbia funzionato stampando le prime righe e
# ricontrollo le lunghezze. il risultato [6] conferma che ora tutti
# i codici hanno 6 cifre

0    001001
1    001001
2    001001
3    001001
4    001001
5    001001
6    001001
7    001001
8    001001
9    001001
Name: REF_AREA, dtype: str
[6]


In [6]:
df[df['DATA_TYPE'] == 'ROADACC']['OBS_VALUE'].describe()

# guardando i numeri di .describe() si vede che qualcosa non è regolare:
# la mediana (50%) è 3, ma la media è quasi 25 (molto più alta della
# mediana). anche la deviazione standard (257) è enorme, circa 10 volte
# la media stessa. quando media e mediana sono così distanti, e la
# deviazione standard è più grande della media, di solito vuol dire che
# c'è almeno un valore molto più alto degli altri che tira su la media
# rispetto al resto dei dati, cioè un possibile outlier

count    191184.000000
mean         24.997599
std         257.519973
min           0.000000
25%           1.000000
50%           3.000000
75%          12.000000
max       23135.000000
Name: OBS_VALUE, dtype: float64

In [7]:
df[(df['DATA_TYPE'] == 'ROADACC') & (df['OBS_VALUE'] == 23135)]

# vado a vedere a quale riga appartiene il valore più alto (23135, il max
# uscito da .describe()), per capire se è un errore nei dati o un valore legittimo
# il codice REF_AREA che esce è 058091, che cercando su internet, corrisponde a Roma Capitale:
# essendo il comune più popoloso d'Italia, ha senso che abbia molti più
# incidenti rispetto agli altri comuni solo per via del traffico e delle dimensioni

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
348636,IT1:41_983(1.0),A,058091,ROADACC,9,2004,23135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df = df.drop(columns=['OBS_STATUS', 'NOTE_DS', 'NOTE_REF_AREA', 'NOTE_DATA_TYPE',
                       'NOTE_RESULT', 'NOTE_TIME_PERIOD', 'BASE_PER', 'UNIT_MEAS', 'UNIT_MULT', 'FREQ', 'DATAFLOW'])

# faccio pulizia togliendo queste colonne dal dataset, ovvero tutte quelle con NaN e le due
# colonne (DATAFLOW e FREQ) che non apportano alcuna informazione utile

In [9]:
df.info()

# controllo con .info() che il drop abbia funzionato: ora il dataframe
# ha 5 colonne invece di 16, tutte piene al 100% (573.552 valori
# non-null su ognuna), e la memoria occupata è scesa da 70 MB a 22 MB

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   REF_AREA     573552 non-null  str  
 1   DATA_TYPE    573552 non-null  str  
 2   RESULT       573552 non-null  str  
 3   TIME_PERIOD  573552 non-null  int64
 4   OBS_VALUE    573552 non-null  int64
dtypes: int64(2), str(3)
memory usage: 21.9 MB


In [10]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M')]['OBS_VALUE'].describe())
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == '9')]['OBS_VALUE'].describe())

# faccio lo stesso controllo .describe() già fatto per ROADACC, ma
# questa volta sui feriti (RESULT=F), sui morti (RESULT=M) e sul 9 che non so ancora cosa sia:
# - F: media 35, mediana 5, max 30254 -> stessa forma fortemente
#   asimmetrica vista per ROADACC, pochi casi enormi tirano su la media
# - M: media 0.53, mediana 0 -> più della metà dei comuni/anni ha
#   zero morti, ha senso perché morire in un incidente è un evento raro.
#   il massimo però è 363, un numero alto, da verificare prima di darlo per buono
# - 9: dalla cella n° 7 del controllo sul massimo di Roma ho già visto che
#   RESULT=9 compare insieme a DATA_TYPE=ROADACC (REF_AREA=058091, TIME_PERIOD=2004).
#   ho anche dimostrato che il conteggio delle righe con DATA_TYPE=KILLINJ e RESULT=9 insieme
#   è zero (count=0), quindi quella combinazione non esiste mai. dato che DATA_TYPE ha solo
#   due valori possibili (KILLINJ, ROADACC, verificato con .unique()), e KILLINJ è escluso,
#   resta confermato che il 9 accompagna sempre e solo ROADACC, il che lo rende un placeholder. il 9
#   è lì solo per dire "l'evento è avvenuto" ma non tiene traccia del numero di morti e feriti in sè
# il perchè ho detto che F=feriti ed M=morti lo spiego più avanti

count    191184.000000
mean         35.091985
std         338.109881
min           0.000000
25%           1.000000
50%           5.000000
75%          18.000000
max       30254.000000
Name: OBS_VALUE, dtype: float64
count    191184.000000
mean          0.533225
std           2.783505
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         363.000000
Name: OBS_VALUE, dtype: float64
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: OBS_VALUE, dtype: float64


# Cercando online ho trovato che ISTAT usa convenzionalmente F per feriti
l'azienda OnData, che lavora a stretto contatto coi df dell'ISTAT, ha postato su GitHub
(fonte: https://github.com/ondata/guida-api-istat)
una guida su come giostrarsi tra i dati ISTAT e nel 3° esempio recita questo:

"Scarica solo i feriti negli incidenti stradali a Palermo, ultimi 10 record disponibili:

curl -kL -H "Accept: application/vnd.sdmx.data+csv;version=1.0.0" \
  "https://esploradati.istat.it/SDMXWS/rest/data/41_983/A.082053.KILLINJ.F?lastNObservations=10""

dove hanno applicato filtri dimensionali: A.082053.KILLINJ.F
- A = frequenza Annuale
- 082053 = codice comune Palermo
- KILLINJ = tipo dato (killed/injured)
- F = feriti
- Richiesto ultimi 10 record con lastNObservations=10 (in questo caso 10 anni perché frequenza è annuale)

per esclusione, visto che F=feriti e il 9 è un placeholder, M=morti


In [11]:
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'F') & (df['OBS_VALUE'] == 30254)])
print(df[(df['DATA_TYPE'] == 'KILLINJ') & (df['RESULT'] == 'M') & (df['OBS_VALUE'] == 363)])

# controllo a chi appartengono i due valori massimi trovati prima
# (30254 feriti, 363 morti): in entrambi i casi esce lo stesso comune,
# REF_AREA=058091 quindi Roma Capitale, lo steso identico comune che era
# già uscito come massimo per ROADACC. quindi è lo stesso fenomeno di
# prima: essendo il comune più popoloso e con più traffico d'Italia,
# ha senso che abbia i numeri più alti in assoluto su tutte le metriche

       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348588   058091   KILLINJ      F         2004      30254
       REF_AREA DATA_TYPE RESULT  TIME_PERIOD  OBS_VALUE
348610   058091   KILLINJ      M         2002        363


In [12]:
df.groupby('TIME_PERIOD')['REF_AREA'].nunique()

# conto quanti comuni distinti ci sono per ogni anno, per vedere se la
# copertura è uniforme nel tempo o se ci sono buchi. dal 2001 al 2023 il
# calo è lento e graduale (da 8101 a 7901 comuni, circa -2.5% in 22 anni) -
# plausibile, è l'effetto delle fusioni di comuni che l'italia ha fatto
# negli ultimi decenni. ma dal 2023 al 2024 il calo è molto più brusco perchè va
# da 7901 a 6339 comuni, -1562 in un solo anno (-20%), troppo per essere
# spiegato dalle normali fusioni: qui serve un controllo più approfondito
# (che faccio nelle celle successive)

TIME_PERIOD
2001    8101
2002    8102
2003    8102
2004    8100
2005    8101
2006    8101
2007    8101
2008    8101
2009    8100
2010    8094
2011    8092
2012    8092
2013    8092
2014    8059
2015    8047
2016    8000
2017    7978
2018    7954
2019    7915
2020    7904
2021    7904
2022    7904
2023    7901
2024    6339
Name: REF_AREA, dtype: int64

In [13]:
print(df[df['TIME_PERIOD'] == 2023]['REF_AREA'].nunique())
print(df[df['TIME_PERIOD'] == 2024]['REF_AREA'].nunique())

# verifico se il calo visto sopra nel conteggio delle righe per anno
# riguarda intere righe di comuni mancanti, o solo singole metriche: conto
# i comuni distinti (REF_AREA) per il 2023 e il 2024. sono 7901
# comuni nel 2023, e solo 6339 nel 2024. una perdita di 1562 comuni interi,
# circa il 20%, come già notato nel conteggio totale delle righe.
# questo conferma che il 2024 ha dati probabilmente incompleti
# (mancano intere rilevazioni comunali), non solo "meno incidenti
# registrati" e per questo più avanti escluderò questo anno dal dataset princiale

7901
6339


In [14]:
df_completo = df.copy()
df = df_completo[df_completo['TIME_PERIOD'] != 2024].copy()

# tengo una copia con tutti gli anni (df_completo), incluso il 2024,
# si sa mai che dovesse servirmi, ed escludo il 2024 dal dataset principale (df).

# Ora che il dataset è pulito, procedo a fare la merge con SITUAS

In [15]:
df_pivot = df.pivot_table(index=['REF_AREA' , 'TIME_PERIOD'], columns=['DATA_TYPE' , 'RESULT'], values='OBS_VALUE')

# controllo che l'ordine delle colonne sia quello atteso PRIMA di rinominare
# a mano nella cella dopo: se l'ordine cambiasse, le etichette finirebbero
# sulle colonne sbagliate senza dare errore
assert list(df_pivot.columns) == [('KILLINJ', 'F'), ('KILLINJ', 'M'), ('ROADACC', '9')]

df_pivot.head()

# trasformo il dataset con pivot_table:
# REF_AREA e TIME_PERIOD diventano l'indice (una riga per ogni comune+anno),
# DATA_TYPE e RESULT insieme diventano le nuove colonne, e OBS_VALUE i
# valori dentro quelle colonne

# dal .columns si vede che le colonne sono un MultiIndex, cioè identificate
# da coppie di valori e non da un nome singolo: ('KILLINJ', 'F'),
# ('KILLINJ', 'M'), ('ROADACC', '9'), sono le stesse tre combinazioni
# che avevo già trovato controllando i valori unici di DATA_TYPE e RESULT

DATA_TYPE            KILLINJ      ROADACC
RESULT                     F    M       9
REF_AREA TIME_PERIOD                     
001001   2001           10.0  0.0     5.0
         2002           10.0  0.0     5.0
         2003            7.0  0.0     4.0
         2004           13.0  0.0     9.0
         2005            2.0  0.0     2.0

In [16]:
df_pivot.columns = ['feriti', 'morti', 'incidenti']

# ho rinominayo le colonne del MultiIndex in nomi semplici e in italiano,
# rispettando lo stesso ordine visto sopra con .columns:
# ('KILLINJ', 'F') -> feriti, ('KILLINJ', 'M') -> morti,
# ('ROADACC', '9') -> incidenti

df_pivot

feriti  morti  incidenti
REF_AREA TIME_PERIOD                          
001001   2001           10.0    0.0        5.0
         2002           10.0    0.0        5.0
         2003            7.0    0.0        4.0
         2004           13.0    0.0        9.0
         2005            2.0    0.0        2.0
...                      ...    ...        ...
111107   2019            5.0    0.0        5.0
         2020            3.0    0.0        2.0
         2021            7.0    0.0        5.0
         2022            1.0    0.0        1.0
         2023            5.0    0.0        4.0

[184845 rows x 3 columns]

In [17]:
df_pivot.isnull().sum()

# controllo se ci sono valori mancanti (NaN) nel pivot, su tutte le
# combinazioni comune+anno, non ci sono,
# quindi ogni comune/anno ha sempre un valore registrato per tutte le
# metriche (anche se a volte quel valore è 0, es. per i morti nei
# comuni più piccoli). posso quindi convertire le colonne in int senza
# preoccuparmi di buchi da gestire prima

feriti       0
morti        0
incidenti    0
dtype: int64

In [18]:
df_pivot = df_pivot.astype(int)

# convertto le colonne da float a int: non è strettamente necessario
# ma essendo conteggi (incidenti, feriti, morti)
# ha più senso rappresentarli come numeri interi, ed è anche
# più leggibile da guardare rispetto ad avere ".0" ovunque. il pivot li
# aveva trasformati in float per gestire eventuali NaN, ma avendo già
# verificato sopra che non ce ne sono, posso convertire senza perdere nulla

In [19]:
df_pivot = df_pivot.reset_index()
df_pivot

# riporto REF_AREA e TIME_PERIOD a essere colonne normali invece che
# indice del dataframe (così come erano impostate dal pivot_table),
# per prepararmi al join con i dati SITUAS più avanti, che funziona
# meglio con colonne standard piuttosto che con l'indice a due livelli.
# al loro posto pandas crea un nuovo indice numerico sequenziale

# ho notato che nel modificare le celle precedenti per ricontrollare tutto,
# se faccio partire le celle a singhiozzo mi crea un secondo reset_index() aggiungendo un'altra colonna inutile
# chiamata proprio 'index' che nella join potrebbe rompere le scatole, e inoltre mi ha pure tolto
# gli zeri davanti ai record di REF_AREA. ho sistemato questa cosa molto semplicemente restartando il kernel

,REF_AREA,TIME_PERIOD,feriti,morti,incidenti
0,001001,2001,10,0,5
1,001001,2002,10,0,5
2,001001,2003,7,0,4
3,001001,2004,13,0,9
4,001001,2005,2,0,2
...,...,...,...,...,...
184840,111107,2019,5,0,5
184841,111107,2020,3,0,2
184842,111107,2021,7,0,5
184843,111107,2022,1,0,1


In [20]:
df_pivot = df_pivot.rename(columns={'REF_AREA': 'codice_comune', 'TIME_PERIOD': 'anno'})
df_pivot.head()

# rinomino le colonne tecniche dell'ISTAT con nomi italiani come prima:
# 'REF_AREA' diventa 'codice_comune', 'TIME_PERIOD' diventa 'anno'.
# uso rename con un dizionario perché lavora per nome e non per posizione,
# quindi rinomina le colonne ovunque si trovino ed è immune a eventuali
# cambi di ordine. qui posso usarlo perché REF_AREA e TIME_PERIOD sono già
# colonne normali e piatte avendo usato il reset_index, devo solo cambiare
# l'etichetta. prima invece, sulle colonne del pivot, non potevo usare il
# dizionario perché quelle erano un MultiIndex e dovevo appiattirlo in nomi
# singoli (cambio di struttura, non di solo nome) ecco perchè avevo usato
# l'assegnazione posizionale df.columns = [...], che però è cieca all'ordine
# e per questo l'avevo blindata con l'assert.
# chiamo la colonna codice_comune (e non comune) di proposito, perché
# contiene il CODICE del comune (es. 001001) e non il nome: più avanti da
# SITUAS porto dentro anche la colonna col nome vero, così le due cose
# restano distinte

,codice_comune,anno,feriti,morti,incidenti
0,001001,2001,10,0,5
1,001001,2002,10,0,5
2,001001,2003,7,0,4
3,001001,2004,13,0,9
4,001001,2005,2,0,2


## Verifico se un solo anno di popolazione è una semplificazione accettabile

Il README specifica che i dati SITUAS sono statici e che è accettabile scaricarli una sola volta ("it's fine to download it manually once"), ma avvisa anche che il csv è specifico per anno, mentre il dataset incidenti copre 23 anni (2001-2023).

Prima di decidere se scaricare un solo anno o tutti, verifico quanto varia la popolazione dei comuni tra il 2001 e il 2023. NON guardo la variazione *media* nazionale, che sarebbe fuorviante (l'Italia nel complesso è demograficamente stabile, ma questo nasconde i singoli comuni che crescono o si spopolano molto). Guardo invece la *distribuzione* delle variazioni comune per comune: quanti comuni cambiano poco e quanti cambiano molto, e qual è il caso peggiore. Se la grande maggioranza dei comuni varia di poco e i casi estremi (gli outlier) sono rari, allora un solo anno è una semplificazione ragionevole; se invece molti comuni cambiano in modo marcato, conviene usare il dato annuale, perché usare una popolazione sbagliata falserebbe il calcolo del tasso di incidenti pro capite proprio sui comuni più dinamici.

In [21]:
import os
cartella = 'CSV Situas'

pop_2001 = pd.read_csv(os.path.join(cartella, 'Comuni - Dimensione Data Indagine 31-12-2001.csv'),
                       sep=';', decimal=',', thousands='.')
pop_2023 = pd.read_csv(os.path.join(cartella, 'Comuni - Dimensione Data Indagine 31-12-2023.csv'),
                       sep=';', decimal=',', thousands='.')

# carico entrambi i file SITUAS (popolazione 2001 e 2023) in un'unica cella,
# applicando direttamente tutti i parametri necessari invece di scoprirli a
# tentativi separati. i CSV sono in formato italiano, quindi uso sep=';' come
# separatore di campo, decimal=',' perché la virgola è il separatore decimale
# (senza, la Superficie come 13,1462 verrebbe letta come stringa e i calcoli di
# densità si romperebbero, mentre così diventa float), e thousands='.' perché
# il punto raggruppa le migliaia nelle popolazioni delle città grandi. la
# verifica dei tipi e del formato la faccio nella cella successiva.

In [22]:
pop_2001 = pop_2001[['Codice Comune (alfanumerico)', 'Comune', 'Sigla automobilistica',
                      'Codice Provincia/Uts', 'Superficie (Kmq)', 'Popolazione residente']]
pop_2023 = pop_2023[['Codice Comune (alfanumerico)', 'Comune', 'Sigla automobilistica',
                      'Codice Provincia/Uts', 'Superficie (Kmq)', 'Popolazione residente']]

# tengo solo le colonne che mi servono da entrambi i file: il codice comune,
# il nome del comune, la sigla automobilistica, il codice provincia, la
# superficie in kmq (necessaria per calcolare la densità di incidenti per
# km² richiesta dal README) e la popolazione residente (necessaria per il
# calcolo pro capite). nel file 2023 c'è anche una colonna
# "Codice Provincia (Storico)", che non prendo perché mi serve quello
# attuale (Codice Provincia/Uts)

In [23]:
pop_2001

# stampo una per una per vedere il nome delle colonne come indice

,Codice Comune (alfanumerico),Comune,Sigla automobilistica,Codice Provincia/Uts,Superficie (Kmq),Popolazione residente
0,1001,Agliè,TO,1,13.1462,2557
1,1002,Airasca,TO,1,15.7393,3543
2,1003,Ala di Stura,TO,1,46.3315,480
3,1004,Albiano d'Ivrea,TO,1,11.7315,1687
4,1005,Alice Superiore,TO,1,7.3796,619
...,...,...,...,...,...,...
8097,103073,Viganella,VB,103,13.6649,204
8098,103074,Vignone,VB,103,3.3806,1087
8099,103075,Villadossola,VB,103,18.7333,6898
8100,103076,Villette,VB,103,7.3834,241


In [24]:
pop_2023

# ho verificato il salvataggio corretto dei cambiamenti delle colonne

,Codice Comune (alfanumerico),Comune,Sigla automobilistica,Codice Provincia/Uts,Superficie (Kmq),Popolazione residente
0,1001,Agliè,TO,201,13.1463,2596
1,1002,Airasca,TO,201,15.7393,3686
2,1003,Ala di Stura,TO,201,46.3316,472
3,1004,Albiano d'Ivrea,TO,201,11.7397,1617
4,1006,Almese,TO,201,17.8741,6315
...,...,...,...,...,...,...
7895,111103,Villaputzu,SU,111,181.3947,4438
7896,111104,Villasalto,SU,111,130.3596,938
7897,111105,Villasimius,SU,111,58.1781,3721
7898,111106,Villasor,SU,111,86.8137,6600


In [25]:
pop_2001['Codice Comune (alfanumerico)'] = pop_2001['Codice Comune (alfanumerico)'].astype(str).str.zfill(6)
pop_2023['Codice Comune (alfanumerico)'] = pop_2023['Codice Comune (alfanumerico)'].astype(str).str.zfill(6)

# riporto il codice comune a stringa di 6 cifre con zfill come in precedneza
# perchè pandas lo legge come intero perdendo gli zeri (1001 invece di 001001)
# come ha fatto prima, se no non combacerebbe con la chiave degli incidenti nel merge

In [26]:
pop_2001 = pop_2001.rename(columns={'Popolazione residente': 'pop_2001'})
pop_2023 = pop_2023.rename(columns={'Popolazione residente': 'pop_2023'})

# rinomino la colonna popolazione in ognuno dei due dataframe, perché
# altrimenti dopo il merge avrei due colonne con lo stesso nome
# (Popolazione residente) e non potrei distinguerle

In [27]:
confronto_pop = pop_2001[['Codice Comune (alfanumerico)', 'pop_2001']].merge(
    pop_2023[['Codice Comune (alfanumerico)', 'pop_2023']],
    on='Codice Comune (alfanumerico)'
)
confronto_pop

# adesso che ho sistemato i df, per decidere se un solo anno di popolazione basta, costruisco un dataframe
# di confronto unendo SOLO il codice comune e le due popolazioni,
# senza trascinarmi dietro superficie, nome e sigla, che tanto
# non mi servono per questa verifica. uso un merge usando Codice
# Comune come PK così ogni riga avrà la popolazione del comune nei due anni

,Codice Comune (alfanumerico),pop_2001,pop_2023
0,001001,2557,2596
1,001002,3543,3686
2,001003,480,472
3,001004,1687,1617
4,001006,5648,6315
...,...,...,...
7537,103072,30066,30024
7538,103074,1087,1186
7539,103075,6898,6238
7540,103076,241,277


In [28]:
confronto_pop['tasso_variazione'] = (
    (confronto_pop['pop_2023'] - confronto_pop['pop_2001']) / confronto_pop['pop_2001'] * 100
).round(2)
confronto_pop

# calcolo il tasso di variazione della popolazione di ogni comune tra il 2001
# e il 2023: (valore finale - valore iniziale) / valore iniziale * 100 e
# arrotondo a 2 decimali per favorire la leggibilità del dato.
# positivo = crescita e negativo = spopolamento

,Codice Comune (alfanumerico),pop_2001,pop_2023,tasso_variazione
0,001001,2557,2596,1.53
1,001002,3543,3686,4.04
2,001003,480,472,-1.67
3,001004,1687,1617,-4.15
4,001006,5648,6315,11.81
...,...,...,...,...
7537,103072,30066,30024,-0.14
7538,103074,1087,1186,9.11
7539,103075,6898,6238,-9.57
7540,103076,241,277,14.94


In [29]:
confronto_pop['tasso_variazione_assoluto'] = confronto_pop['tasso_variazione'].abs()
confronto_pop.head()

# con .abs() tolgo il segno al tasso di variazione (es. -9.57 diventa 9.57).
# mi serve perché voglio sapere QUANTO un comune è cambiato, non se è cresciuto
# o calato: sbaglio il calcolo pro capite del 30% sia se un comune cresce del
# 30% sia se cala del 30%. senza togliere il segno, crescite e cali si
# annullerebbero a vicenda e nasconderebbero il cambiamento reale

,Codice Comune (alfanumerico),pop_2001,pop_2023,tasso_variazione,tasso_variazione_assoluto
0,001001,2557,2596,1.53,1.53
1,001002,3543,3686,4.04,4.04
2,001003,480,472,-1.67,1.67
3,001004,1687,1617,-4.15,4.15
4,001006,5648,6315,11.81,11.81


In [30]:
print(confronto_pop['tasso_variazione_assoluto'].describe())
print("\ndecili (ogni 10%):")
print(confronto_pop['tasso_variazione_assoluto'].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]))

# guardo la distribuzione del tasso di variazione assoluto con i decili, non
# la media (che nasconderebbe gli estremi) così vedo quanti comuni cambiano
# poco e quanti tanto. se la maggior parte sta su valori bassi e ci sono pochi outier,
# un solo anno di popolazione è una semplificazione accettabile

count    7542.000000
mean       14.676958
std        12.681441
min         0.000000
25%         5.640000
50%        11.950000
75%        20.540000
max       266.670000
Name: tasso_variazione_assoluto, dtype: float64

decili (ogni 10%):
0.1      2.170
0.2      4.512
0.3      6.800
0.4      9.210
0.5     11.950
0.6     14.946
0.7     18.560
0.8     23.058
0.9     29.949
1.0    266.670
Name: tasso_variazione_assoluto, dtype: float64


In [31]:
idx_max = confronto_pop['tasso_variazione'].idxmax()
confronto_pop.loc[idx_max]

# uso .idxmax() per trovare il comune con la crescita più alta,
# sull'internet mi esce che 018127 corrisponde a Rognano, in provincia di Pavia

Codice Comune (alfanumerico)    018127
pop_2001                           192
pop_2023                           704
tasso_variazione                266.67
tasso_variazione_assoluto       266.67
Name: 2376, dtype: object

## Un solo anno non basta e le fasce sarebbero scomode, li uso tutti

La verifica ha avuto successo e ha fatto quelo che doveva fare, la mediana del tasso di variazione è dell'11,95% quindi metà dei comuni ha cambiato popolazione di almeno il 12% dal 2001 al 2023, il 70% dei comuni sta sotto il 18%, ma il restante 30% va anche oltre, fino a un comune che è quasi quadruplicato (266%).

Avevo inizialmente pensato di suddividere gli anni in fasce (6 fasce da 4 anni ciascuno + 1 fascia da 3 anni) e prendere gli anni in mezzo a queste fasce, e usarli per fare i calcoli successivi ma a sto punto, per evitare di sbilanciare i dati di poco ma sia da una parte che dall'altra rispetto agli estremi di ogni fascia, prendo tutti gli anni da SITUAS e la faccio finita così.

In [32]:
import os
cartella = 'CSV Situas'
for anno in range(2001, 2024):
    f = os.path.join(cartella, f'Comuni - Dimensione Data Indagine 31-12-{anno}.csv')
    print(anno, "OK" if os.path.exists(f) else "MANCA")

# controllo che i file che ho scaricato ci siano tutti

2001 OK
2002 OK
2003 OK
2004 OK
2005 OK
2006 OK
2007 OK
2008 OK
2009 OK
2010 OK
2011 OK
2012 OK
2013 OK
2014 OK
2015 OK
2016 OK
2017 OK
2018 OK
2019 OK
2020 OK
2021 OK
2022 OK
2023 OK


In [33]:
import os
cartella = 'CSV Situas'

lista_popolazioni = []
for anno in range(2001, 2024):
    f = os.path.join(cartella, f'Comuni - Dimensione Data Indagine 31-12-{anno}.csv')
    pop = pd.read_csv(f, sep=';', decimal=',', thousands='.')
    pop = pop[['Codice Comune (alfanumerico)', 'Comune', 'Sigla automobilistica',
               'Superficie (Kmq)', 'Popolazione residente']].copy()
    pop['Codice Comune (alfanumerico)'] = pop['Codice Comune (alfanumerico)'].astype(str).str.zfill(6)
    pop['anno'] = anno
    lista_popolazioni.append(pop)

pop_completo = pd.concat(lista_popolazioni, ignore_index=True)
pop_completo

# carico tutti i 23 file SITUAS (2001-2023) con un ciclo che li legge dalla
# cartella 'CSV Situas'. a ogni file applico la stessa pulizia: separatori
# italiani (sep, decimal, thousands), selezione delle sole colonne utili e
# zfill sul codice comune. aggiungo a ognuno una colonna 'anno' e li accumulo
# in lista_popolazioni, poi li impilo tutti con concat in un unico dataframe

,Codice Comune (alfanumerico),Comune,Sigla automobilistica,Superficie (Kmq),Popolazione residente,anno
0,001001,Agliè,TO,13.1462,2557,2001
1,001002,Airasca,TO,15.7393,3543,2001
2,001003,Ala di Stura,TO,46.3315,480,2001
3,001004,Albiano d'Ivrea,TO,11.7315,1687,2001
4,001005,Alice Superiore,TO,7.3796,619,2001
...,...,...,...,...,...,...
184830,111103,Villaputzu,SU,181.3947,4438,2023
184831,111104,Villasalto,SU,130.3596,938,2023
184832,111105,Villasimius,SU,58.1781,3721,2023
184833,111106,Villasor,SU,86.8137,6600,2023


In [37]:
pop_completo = pop_completo.rename(columns={
    'Codice Comune (alfanumerico)': 'codice_comune',
    'Comune': 'comune',
    'Sigla automobilistica': 'sigla',
    'Superficie (Kmq)': 'superficie_kmq',
    'Popolazione residente': 'popolazione'
})
pop_completo

# rinomino le colonne di pop_completo in nomi puliti e coerenti con df_pivot,
# 'Codice Comune (alfanumerico)' diventa 'codice_comune', lo
# stesso nome che ho nel df degli incidenti, così il merge aggancia le due
# tabelle sulla stessa chiave. uso rename con dizionario (lavora per nome,
# immune all'ordine)

,codice_comune,comune,sigla,superficie_kmq,popolazione,anno
0,001001,Agliè,TO,13.1462,2557,2001
1,001002,Airasca,TO,15.7393,3543,2001
2,001003,Ala di Stura,TO,46.3315,480,2001
3,001004,Albiano d'Ivrea,TO,11.7315,1687,2001
4,001005,Alice Superiore,TO,7.3796,619,2001
...,...,...,...,...,...,...
184830,111103,Villaputzu,SU,181.3947,4438,2023
184831,111104,Villasalto,SU,130.3596,938,2023
184832,111105,Villasimius,SU,58.1781,3721,2023
184833,111106,Villasor,SU,86.8137,6600,2023


In [35]:
df_finale = df_pivot.merge(pop_completo, on=['codice_comune', 'anno'], how='left')
df_finale

# qui unisco gli incidenti (df_pivot) con la popolazione (pop_completo).
# uso due chiavi insieme, codice_comune e anno, perché ogni comune compare
# in tutti i 23 anni: se unissi solo sul codice, ogni anno di incidenti si
# attaccherebbe a tutti gli anni di popolazione facendo un casino. con la
# coppia codice+anno invece il 2015 di un comune si lega solo al 2015 dello
# stesso comune. metto how='left' partendo dagli incidenti così non ne perdo nessuno

,codice_comune,anno,feriti,morti,incidenti,comune,sigla,superficie_kmq,popolazione
0,001001,2001,10,0,5,Agliè,TO,13.1462,2557.0
1,001001,2002,10,0,5,Agliè,TO,13.1462,2538.0
2,001001,2003,7,0,4,Agliè,TO,13.1462,2588.0
3,001001,2004,13,0,9,Agliè,TO,13.1462,2679.0
4,001001,2005,2,0,2,Agliè,TO,13.1462,2674.0
...,...,...,...,...,...,...,...,...,...
184840,111107,2019,5,0,5,Villaspeciosa,SU,27.1937,2605.0
184841,111107,2020,3,0,2,Villaspeciosa,SU,27.1937,2549.0
184842,111107,2021,7,0,5,Villaspeciosa,SU,27.1943,2536.0
184843,111107,2022,1,0,1,Villaspeciosa,SU,27.1943,2575.0


In [36]:
print("NaN per colonna:")
print(df_finale.isnull().sum())
print("\nrighe totali:", len(df_finale))
print("righe con popolazione mancante:", df_finale['popolazione'].isnull().sum())

# col print precedente mi è saltato all'occhio che la popolazione ha preso dei numeri decimali,
# pandas ha trovato dei NaN e li ha messi dove non c'è corrispondenza, verfico sta cosa
# cercando e contando i NaN per ogni colonna

NaN per colonna:
codice_comune        0
anno                 0
feriti               0
morti                0
incidenti            0
comune             307
sigla             2400
superficie_kmq     284
popolazione        284
dtype: int64

righe totali: 184845
righe con popolazione mancante: 284


In [ ]:
righe_nan = df_finale[df_finale['popolazione'].isnull()]
print("incidenti nelle righe nan:", righe_nan['incidenti'].sum(), "su", df_finale['incidenti'].sum())

# prima di buttare le 284 righe senza popolazione voglio sapere quanto mi
# costano davvero, cioè quanti incidenti ci sono dentro. se sono una manciata
# rispetto al totale allora toglierle non cambia niente e posso
# farlo tranquillo, se invece pesano tanto devo pensarci meglio

incidenti nelle righe nan: 6187 su 4605777


In [39]:
df_finale = df_finale[df_finale['popolazione'].notnull()]
print(df_finale.shape)
print(df_finale.isnull().sum())

# tengo solo le righe che hanno la popolazione, togliendo le 284 senza
# corrispondenza in SITUAS (comuni fusi o ricodificati). dentro c'erano 6187
# incidenti su 4.605.777, lo 0,134%, quota irrilevante. senza popolazione non
# potrei calcolare né densità né tasso pro capite, quindi tenerle sarebbe inutile

(184561, 9)
codice_comune        0
anno                 0
feriti               0
morti                0
incidenti            0
comune              23
sigla             2116
superficie_kmq       0
popolazione          0
dtype: int64
